In [1]:
# Start with tinyshakespeare dataset to follow Karpathy for initial development
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-09 13:47:25--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M   699KB/s    in 1.6s    

2026-05-09 13:47:27 (699 KB/s) - ‘input.txt’ saved [1115394/1115394]



In [6]:
s = set("aefasdjfkafjslkdflajdf")
len(s)

for i,c in enumerate(s):
    print(f"{i}:{c}")

0:d
1:j
2:e
3:a
4:l
5:k
6:s
7:f


In [ ]:
# Simple char based tokenization, attempting compatibility with tiktoken methods for later drop in replacement

class CharTokenizer():
    def __init__(self, intext: str):
        self.charset = set(intext)
        sorted_charset = sorted(list(self.charset))
        self.stoi = {c:i for i,c in enumerate(sorted_charset)}
        self.itos = {i:c for i,c in enumerate(sorted_charset)}
        self.n_vocab = len(self.charset)
    
    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]
    
    def decode(self, l: list[int]) -> str:
        return ''.join(self.itos[i] for i in l)


with open("input.txt", 'r', encoding="utf-8") as f:
    text = f.read()
tokenizer = CharTokenizer(text)
print(tokenizer.n_vocab())
print(repr(''.join(sorted(list(tokenizer.charset)))))
s = "Hello"
assert(tokenizer.decode(tokenizer.encode(s))==s)


65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


In [78]:
import torch 

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

# split into train and validation data
split_index = int(0.9*len(data))
train = data[:split_index]
val = data[split_index:]

In [96]:
device = 'mps' if torch.mps.is_available() else 'cpu'

torch.manual_seed(1337)
# batch_size = 4
# block_size = 8

def batch(split, batch_size, block_size):
    if split == 'train':
        data = train
    elif split == 'val':
        data = val
    inds = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds])
    y = torch.stack([data[i+1:i+block_size+1] for i in inds])
    x,y = x.to(device), y.to(device)
    return x, y

# x_train, y_train = batch(train)
# print(x_train.shape)
# print(x_train)
# print(y_train.shape)
# print(y_train)

In [91]:
import math
from einops import rearrange
from torch import einsum
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    
    def __init__(self, block_size, embedding_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads

        self.qry_weights = nn.Parameter(torch.randn(embedding_dim, embedding_dim) / math.sqrt(embedding_dim))
        self.key_weights = nn.Parameter(torch.randn(embedding_dim, embedding_dim) / math.sqrt(embedding_dim))
        self.val_weights = nn.Parameter(torch.randn(embedding_dim, embedding_dim) / math.sqrt(embedding_dim))
        self.out_weights = nn.Parameter(torch.randn(embedding_dim, embedding_dim) / math.sqrt(embedding_dim))

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    
    def forward(self, x):
        #x.shape is (b,l,e) - b is batch size, l is sequence length, e is embedding vector length
        b,l,e = x.shape

        q = einsum('b l e, e d -> b l d', x, self.qry_weights)
        k = einsum('b l e, e d -> b l d', x, self.key_weights)
        v = einsum('b l e, e d -> b l d', x, self.val_weights)

        q = rearrange(q, 'b l (n d) -> b n l d', n=self.num_heads)
        k = rearrange(k, 'b l (n d) -> b n l d', n=self.num_heads)
        v = rearrange(v, 'b l (n d) -> b n l d', n=self.num_heads)

        attn = einsum('b n q d, b n k d -> b n q k', q, k)

        attn = attn / math.sqrt(q.shape[-1])

        attn = attn.masked_fill(self.tril[:l, :l] == 0, float('-inf'))
        attn = attn.softmax(dim=-1)

        out = einsum('b n q l, b n l d -> b n q d', attn, v)

        out = rearrange(out, 'b n q d -> b q (n d)')
        out = einsum('b l e, e o -> b l o', out, self.out_weights)
        return out

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps

        self.gamma = nn.Parameter(torch.ones(d))
        self.beta = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, unbiased=False, keepdim=True)
        norm = (x - mean) / torch.sqrt(var + self.eps)
        return (self.gamma * norm) + self.beta

In [100]:
class TransformerBlock(nn.Module):

    def __init__(self, block_size, embedding_dim, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(block_size, embedding_dim, num_heads)
        self.norm_attn = LayerNorm(embedding_dim)
        self.norm_ff = LayerNorm(embedding_dim)

        ff_dim = embedding_dim * 4

        self.ff = nn.Sequential(
            nn.Linear(embedding_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embedding_dim)
        )
    
    def forward(self, x):
        attn_out = self.norm_attn(self.attn(x) + x)

        ff_out = self.norm_ff(self.ff(attn_out) + attn_out)

        return ff_out

In [116]:
from torch.nn import functional as F
device = 'mps' if torch.mps.is_available() else 'cpu'

class BasicLanguageModel(nn.Module):
    def __init__(self, n_vocab, block_size, embedding_dim=32, num_heads=4, num_layers=4):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(n_vocab, embedding_dim)
        self.position_embedding_table = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(
            *[TransformerBlock(block_size, embedding_dim, num_heads) for _ in range(num_layers)])
        self.norm_final = LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, n_vocab)

    def forward(self, input_ids, targets=None):
        token_embedding = self.token_embedding_table(input_ids)
        position_embedding = self.position_embedding_table(torch.arange(input_ids.shape[1], device=device))
        # x = token_embedding + position_embedding
        # print(f"token_embedding: {token_embedding.shape}")
        # print(f"position_embedding: {position_embedding.shape}")
        x = self.blocks(token_embedding + position_embedding)
        x = self.norm_final(x)

        logits = self.output(x)

        if targets is None:
            loss = None
        else:
            logits = rearrange(logits, 'b l e -> (b l) e')
            targets = rearrange(targets, 'b l -> (b l)')
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, input_ids, max_new_tokens):
        for _ in range(max_new_tokens):
            input_last_block = input_ids[:, -self.block_size:]
            logits, loss = self(input_last_block)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat((input_ids, next_token), dim=1)
        return input_ids

In [ ]:
from tqdm.auto import tqdm

torch.manual_seed(1337)

# b = batch_size #batch size
# l = block_size # Sequence length, in training can always use block size
# e = 32 # Length of the embedding vector that the vocab gets mapped to

batch_size = 64
block_size = 256
embedding_dim = 128
max_iters = 5000
learning_rate = 1e-3
eval_interval = 1000
eval_iters = 200

model = BasicLanguageModel(
    n_vocab = tokenizer.n_vocab(),
    block_size = block_size,
    embedding_dim=embedding_dim,
    num_heads=4, 
    num_layers=4
)
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = batch(split, batch_size, block_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in tqdm(range(max_iters)):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        tqdm.write(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = batch('train', batch_size, block_size)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



In [125]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(tokenizer.decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


For Gaunt! I bear Piftle,
Or how is my roces for I hear,
That there's common here by feeling that
Do Paulini me; yet I am no ouburn furies
For knowledges thy wind, well I have bound
Deset the truck of reason and troublents fair
Were he it.

FRARINABE:
Why within the gaznaRory, to your coulds's
I till not raise your cried men.

KING EDWARD IV:
But my learn joy, then?

DUKE VINCENTIO:
Have you shall be mine.

KING RICHARD III:
Your know that such like my duke is safel'n ask.

KING EDWARD IV:
And frage forth to me! have her time or ordinar.

DUKE OF AUMERLE:
Where he did this brother's digning eat the way.

NURDSE:
What, my lord.

DUKE OF YORK:
No.

WESTMY K:
John of God Part: 'twas give and watcher disording pity,
Come on, sir; for God sate ittoe bid me friend.

KING LEWIS XI:
Now Loatice out.

ARCIStoOX:
Your brother of merciles his prouds,
I'll silke the remember him:
But that at o' the duke him fortune to a
but incurred him to such a strengthen-more stuffice.

KING RICHARD III:
Take 